In [1]:
import { RecursiveSet as Set, Value } from 'recursive-set';

In [2]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

# Murder in the Family
---
We all know *Alice* and *Bob* from Cryptography.  However, little did
we now about their difficult family relations, which might be the reason for
their secrecy.  

Unfortunately, things have escalated recently:
Murder occurred one evening in the home of their father and mother.  Alice and Bob are the only children of their parents. One member of the family murdered another member, the third member witnessed the crime, and the fourth member was an accessory after the fact.  Furthermore, we know the following:

1. The accessory and the witness were of opposite sex.
2. The oldest member and the witness were of opposite sex.
3. The youngest member and the victim were of opposite sex.
4. The accessory was older than the victim.
5. The father was the oldest member.
6. The murderer was not the youngest member.

**Question:** Which of the four—father, mother, Alice, or Bob—was the murderer?

We use the following variables:
* `FM`, `MM`, `AM`, `BM` are true if the father, the mother, Alice, or Bob is the murderer.
* `FW`, `MW`, `AW`, `BW` are true if the father, the mother, Alice, or Bob is the witness.
* `FV`, `MV`, `AV`, `BV` are true if the father, the mother, Alice, or Bob is the victim.
* `FA`, `MA`, `AA`, `BA` are true if the father, the mother, Alice, or Bob is the accessory.
* `AY` is true if Alice is younger than Bob.

In [3]:
const P = set(
    'FM', 'MM', 'AM', 'BM',
    'FW', 'MW', 'AW', 'BW',
    'FV', 'MV', 'AV', 'BV',
    'FA', 'MA', 'AA', 'BA',
    'AY'
);
P

{AA, AM, AV, AW, AY, BA, BM, BV, BW, FA, FM, FV, FW, MA, MM, MV, MW}


We first construct helper constraints to ensure each person has exactly one role, and each role is assigned to exactly one person.

The function `atLeastOne` takes a set of variables `V ` and returns a propositional formula that is true if and only if at least one variable from the set `V` is true.

In [4]:
function atLeastOne(V: Set<string>): string {
    return Array.from(V).join(' ∨ ');
}

In [5]:
atLeastOne(set('a', 'b', 'c'))

a ∨ b ∨ c


The function `atMostOne` takes a set of variables `V ` and returns a propositional formula that is true if and only if at mots one variable from the set `V` is true.

In [6]:
function atMostOne(V: Set<string>): string {
    const P = V.cartesianProduct(V)
               .filterMap(([p, q]) => p != q,
                          ([p, q]) => `(¬${p} ∨ ¬${q})`);
    return Array.from(P).join(' ∧ ');
}

In [7]:
atMostOne(set('a', 'b', 'c'))

(¬a ∨ ¬b) ∧ (¬a ∨ ¬c) ∧ (¬b ∨ ¬a) ∧ (¬b ∨ ¬c) ∧ (¬c ∨ ¬a) ∧ (¬c ∨ ¬b)


The function `exactlyOne` takes a set of variables `V ` and returns a propositional formula that is true if and only if exactly one variable from the set `V` is true.

In [ ]:
function exactlyOne(V: Set<string>): string {
    "your code here"
}

In [ ]:
exactlyOne(set('a', 'b', 'c'))

In [ ]:
let Formulas = set<string>();
Formulas.add(exactlyOne(set('FM', 'FW', 'FV', 'FA')));
Formulas.add(exactlyOne(set('MM', 'MW', 'MV', 'MA')));
Formulas.add(exactlyOne(set('AM', 'AW', 'AV', 'AA')));
Formulas.add(exactlyOne(set('BM', 'BW', 'BV', 'BA')));
Formulas.add(exactlyOne(set('FM', 'MM', 'AM', 'BM')));
Formulas.add(exactlyOne(set('FW', 'MW', 'AW', 'BW')));
Formulas.add(exactlyOne(set('FV', 'MV', 'AV', 'BV')));
Formulas.add(exactlyOne(set('FA', 'MA', 'AA', 'BA')));

**Fact 1**: The accessory and the witness were of opposite sex.

In [ ]:
const f1 = 'your code here';

**Fact 2 & 5**: The oldest member and the witness were of opposite sex. The father was the oldest member.

In [ ]:
const f2 = 'your code here';

**Fact 3**: The youngest member and the victim were of opposite sex.

In [ ]:
const f3a = 'your code here';
const f3b = 'your code here';

**Fact 4**: The accessory was older than the victim.
Father > Mother > Older Child > Younger Child.

In [ ]:
"your code here"

**Fact 6**: The murderer was not the youngest member.

In [ ]:
"your code here"

In [ ]:
Formulas = Formulas.union(set(f1, f2, f3a, f3b, "your code here"));

In [ ]:
import { parse, Variable, Formula } from './PropositionalLogicParser'

In [ ]:
const Gs = Array.from(Formulas).map(f => parse(f));

The function `evaluate(F, I)` takes a propositional formula $F$ and a propositional variable assignment $I$ and evaluates $F$ using the assignment $I$. We define it below to handle our logic operators.

In [ ]:
function evaluate(f: Formula, I: Set<Variable>): boolean {
    if (typeof f == 'string') { return I.has(f); }
    switch (f[0]) {
        case '⊤': return true;
        case '⊥': return false;
        case '¬': { const [_, g] = f;
                    return !evaluate(g, I); }
        default:  { 
            const [op, g, h] = f;
            const [gs, hs] = [g, h].map(e => evaluate(e, I));
            switch (op) {
                case '∧': return gs && hs;
                case '∨': return gs || hs;
                case '→': return gs <= hs;
                case '↔': return gs == hs;
            }
        }
    }
}

The function `allTrue(Fs, I)` takes a list of propositional formulas `Fs`
and a propositional variable assignment `I`.  It returns `true` only if all formulas from `Fs` are 
`true` given the variable assignment `I`.

In [ ]:
function allTrue(Gs: Formula[], I: Set<Variable>): boolean {
    return Gs.every(f => evaluate(f, I));
}

We use the powerset function to filter all possible subsets of variables representing possible valid assignments:

In [ ]:
const validAssignments = P.powerset().filter(I => allTrue(Gs, I));
validAssignments

Running this model limits the valid assignments to a single solution. 